In [1]:
!pip install --upgrade transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 128.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
!pip install --upgrade "mistral-common[audio]"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 82.0 MB/s eta 0:00:00


In [4]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 65.5 MB/s eta 0:00:00


# Data script

In [ ]:
import os
import pandas as pd
from datasets import load_dataset, Audio
from tqdm import tqdm
import soundfile as sf
import librosa
from huggingface_hub import login

# ---------------- CONFIG ----------------
DATASET_NAME = "ai4bharat/IndicVoices"
LANG = "hindi"
OUT_DIR = "benchmark_data"
TARGET_SAMPLE_RATE = 16000

# Updated Buckets to be slightly more flexible
BUCKETS = {
    "5s": (4, 8),
    "15s": (12, 20),
    "30s": (20, 35)
}
CLIPS_PER_BUCKET = 10

# ----------------------------------------

def prepare_folders():
    for name in BUCKETS:
        os.makedirs(os.path.join(OUT_DIR, name), exist_ok=True)

def get_audio_data(token):
    print(f"Streaming {DATASET_NAME} ({LANG})...")
    login(token)

    # 1. Load and 2. Cast the column immediately
    ds = load_dataset(DATASET_NAME, LANG, split="train", streaming=True)
    ds = ds.cast_column("audio_filepath", Audio(sampling_rate=TARGET_SAMPLE_RATE))

    counts = {k: 0 for k in BUCKETS}
    metadata = []

    # Using enumerate to track progress in the terminal
    for i, example in enumerate(tqdm(ds, desc="Sourcing Audio")):

        # 3. Filter Case-Insensitive
        scenario = example.get("scenario", "").lower()
        if scenario not in ["extempore", "conversation", "conversational"]:
            continue

        # 4. Access the Audio Dictionary
        audio_data = example.get("audio_filepath")
        if not audio_data:
            continue

        y = audio_data["array"]
        sr = audio_data["sampling_rate"]
        duration = len(y) / sr

        # 5. Assign to Bucket
        assigned_bucket = None
        for name, (min_d, max_d) in BUCKETS.items():
            if min_d <= duration <= max_d and counts[name] < CLIPS_PER_BUCKET:
                assigned_bucket = name
                break

        if not assigned_bucket:
            continue

        # 6. Save the File
        file_name = f"clip_{counts[assigned_bucket]}.wav"
        file_path = os.path.join(OUT_DIR, assigned_bucket, file_name)

        sf.write(file_path, y, TARGET_SAMPLE_RATE)

        # 7. Use the CLEAN 'normalized' text for the Ground Truth
        metadata.append({
            "file": file_path,
            "duration": round(duration, 2),
            "bucket": assigned_bucket,
            "text": example.get("normalized", "")
        })

        counts[assigned_bucket] += 1

        # Stop if we are finished
        if all(c >= CLIPS_PER_BUCKET for c in counts.values()):
            break

    # 8. Save the Reference Sheet
    df = pd.DataFrame(metadata)
    df.to_csv(os.path.join(OUT_DIR, "metadata.csv"), index=False)
    print("\n✅ Dataset Sourced! Check the 'benchmark_data' folder.")

if __name__ == "__main__":
    MY_TOKEN = ""
    prepare_folders()
    get_audio_data(MY_TOKEN)

Streaming ai4bharat/IndicVoices (hindi)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/88 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Sourcing Audio: 625it [00:10, 60.89it/s] 


✅ Dataset Sourced! Check the 'benchmark_data' folder.


# Main script



In [5]:
import time
import torch
import pandas as pd
import numpy as np
import jiwer
from threading import Thread

from transformers import VoxtralRealtimeForConditionalGeneration, AutoProcessor, TextIteratorStreamer
from mistral_common.tokens.tokenizers.audio import Audio

# -----------------------------
# CONFIG
# -----------------------------
MODEL_ID = "mistralai/Voxtral-Mini-4B-Realtime-2602"
METADATA_CSV = "benchmark_data/metadata.csv"

DEVICE = "cuda"

# -----------------------------
# TEXT NORMALIZATION (WER)
# -----------------------------
transformation = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.Strip(),
    jiwer.RemoveMultipleSpaces()
])

# -----------------------------
# LOAD MODEL
# -----------------------------
print("🚀 Loading model on GPU...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

model = VoxtralRealtimeForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("✅ Model loaded!\n")

🚀 Loading model on GPU...


processor_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tekken.json:   0%|          | 0.00/14.9M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.86G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/711 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Model loaded!



In [9]:
# TRANSCRIPTION FUNCTION
# -----------------------------
def transcribe_with_latency(audio_path):
    # Load audio
    audio = Audio.from_file(audio_path, strict=False)
    audio.resample(processor.feature_extractor.sampling_rate)

    inputs = processor(audio.audio_array, return_tensors="pt")
    inputs = inputs.to(model.device, dtype=model.dtype)

    # Streamer for TTFT
    streamer = TextIteratorStreamer(processor.tokenizer, skip_special_tokens=True)

    start_time = time.time()
    first_token_time = None

    def generate():
        with torch.inference_mode():
            model.generate(**inputs, streamer=streamer)

    thread = Thread(target=generate)
    thread.start()

    text = ""

    for token in streamer:
        if first_token_time is None:
            first_token_time = time.time()
        text += token

    end_time = time.time()

    ttft = first_token_time - start_time if first_token_time else None
    upl = end_time - start_time

    return text, ttft, upl

# -----------------------------
# BENCHMARK
# -----------------------------
def run_benchmark():
    df = pd.read_csv(METADATA_CSV)
    final_results = []

    print("Starting Voxtral Benchmark")
    print("-" * 50)

    for bucket in ["5s", "15s", "30s"]:
        subset = df[df.bucket == bucket]
        if subset.empty:
            continue

        ttft_list, upl_list, wer_list = [], [], []

        for _, row in subset.head(8).iterrows():
            clip_path, ref_text = row.file, row.text
            print(f"\n[BUCKET: {bucket}] {clip_path}")

            clip_ttft, clip_upl, clip_wer = [], [], []

            for run_idx in range(3):
                try:
                    raw_text, ttft, upl = transcribe_with_latency(clip_path)

                    ttft_ms = ttft * 1000
                    upl_ms = upl

                    clean_ref = transformation(ref_text)
                    clean_hyp = transformation(raw_text)
                    error_rate = jiwer.wer(clean_ref, clean_hyp)

                    clip_ttft.append(ttft_ms)
                    clip_upl.append(upl_ms)
                    clip_wer.append(error_rate)

                    print(f" Run {run_idx+1} | TTFT: {ttft_ms:.3f}ms | UPL: {upl_ms:.3f}s | WER: {error_rate:.3f}")

                except Exception as e:
                    print(f" Run {run_idx+1} Failed: {e}")

            avg_ttft = np.mean(clip_ttft)
            avg_upl = np.mean(clip_upl)
            avg_wer = np.mean(clip_wer)

            ttft_list.append(avg_ttft)
            upl_list.append(avg_upl)
            wer_list.append(avg_wer)

            print(f" Clip Avg | TTFT: {avg_ttft:.3f}ms | UPL: {avg_upl:.3f}s | WER: {avg_wer:.3f}")

        # Percentiles
        p50_ttft, p95_ttft = np.percentile(ttft_list, [50, 95])
        p50_upl, p95_upl = np.percentile(upl_list, [50, 95])
        p50_wer, p95_wer = np.percentile(wer_list, [50, 95])

        final_results.append({
            "Bucket": bucket,
            "Model": "Voxtral",
            "p50_ttft_ms": round(p50_ttft, 3),
            "p95_ttft_ms": round(p95_ttft, 3),
            "p50_upl_s": round(p50_upl, 3),
            "p95_upl_s": round(p95_upl, 3),
            "wer": round(np.mean(wer_list), 3),
            "p50_wer": round(p50_wer, 3),
            "p95_wer": round(p95_wer, 3)
        })

    summary_df = pd.DataFrame(final_results)
    print("\n" + "="*30 + " FINAL SUMMARY " + "="*30)
    print(summary_df.to_string(index=False))

    summary_df.to_csv("voxtral_offline_summary.csv", index=False)
    print("\n✅ Saved to voxtral_offline_summary.csv")

# -----------------------------
# ENTRY POINT
# -----------------------------
if __name__ == "__main__":
    run_benchmark()




Starting Faster Whisper Benchmark
--------------------------------------------------

[BUCKET: 5s] benchmark_data/5s/clip_0.wav


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1554: UserWarning: Using the model-agnostic default `max_length` (=140) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


 Run 1 | TTFT: 14.372ms | UPL: 9.612s | WER: 0.136
 Run 2 | TTFT: 14.457ms | UPL: 7.350s | WER: 0.136
 Run 3 | TTFT: 13.310ms | UPL: 7.423s | WER: 0.136
 Clip Avg | TTFT: 14.047ms | UPL: 8.128s | WER: 0.136

[BUCKET: 5s] benchmark_data/5s/clip_1.wav


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1554: UserWarning: Using the model-agnostic default `max_length` (=109) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


 Run 1 | TTFT: 13.798ms | UPL: 5.108s | WER: 0.056
 Run 2 | TTFT: 13.901ms | UPL: 5.084s | WER: 0.056
 Run 3 | TTFT: 13.216ms | UPL: 5.074s | WER: 0.056
 Clip Avg | TTFT: 13.638ms | UPL: 5.089s | WER: 0.056

[BUCKET: 5s] benchmark_data/5s/clip_2.wav


...

 Run 1 | TTFT: 14.376ms | UPL: 21.603s | WER: 0.236
 Run 2 | TTFT: 13.224ms | UPL: 21.635s | WER: 0.236
 Run 3 | TTFT: 13.266ms | UPL: 21.627s | WER: 0.236
 Clip Avg | TTFT: 13.622ms | UPL: 21.622s | WER: 0.236

[BUCKET: 30s] benchmark_data/30s/clip_5.wav
 Run 1 | TTFT: 14.313ms | UPL: 22.496s | WER: 0.316
 Run 2 | TTFT: 13.241ms | UPL: 22.531s | WER: 0.316
 Run 3 | TTFT: 13.261ms | UPL: 22.288s | WER: 0.316
 Clip Avg | TTFT: 13.605ms | UPL: 22.438s | WER: 0.316

[BUCKET: 30s] benchmark_data/30s/clip_6.wav


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1554: UserWarning: Using the model-agnostic default `max_length` (=320) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


 Run 1 | TTFT: 15.697ms | UPL: 20.542s | WER: 0.404
 Run 2 | TTFT: 13.516ms | UPL: 20.659s | WER: 0.404
 Run 3 | TTFT: 13.815ms | UPL: 20.624s | WER: 0.404
 Clip Avg | TTFT: 14.343ms | UPL: 20.608s | WER: 0.404

[BUCKET: 30s] benchmark_data/30s/clip_7.wav


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1554: UserWarning: Using the model-agnostic default `max_length` (=312) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


 Run 1 | TTFT: 14.310ms | UPL: 19.961s | WER: 0.210
 Run 2 | TTFT: 13.087ms | UPL: 19.970s | WER: 0.210
 Run 3 | TTFT: 13.324ms | UPL: 19.998s | WER: 0.210
 Clip Avg | TTFT: 13.574ms | UPL: 19.976s | WER: 0.210

============================== FINAL SUMMARY ==============================
Bucket          Model  p50_ttft_ms  p95_ttft_ms  p50_upl_s  p95_upl_s   wer  p50_wer  p95_wer
    5s Faster-Whisper       13.766       14.098      6.054      7.575 0.208    0.250    0.311
   15s Faster-Whisper       14.191       15.062     14.879     19.059 0.300    0.216    0.764
   30s Faster-Whisper       13.834       14.759     21.226     22.453 0.227    0.223    0.373

✅ Saved to voxtral_offline_summary.csv
